## Mediapipe 기반 영상 분석 후 csv 데이터 만들기 -> 만들어진 csv 데이터 기반 LLM 프롬프팅만 하여 러닝 자세 분석

In [2]:
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
from dataclasses import dataclass
from mediapipe.tasks import python
from mediapipe.tasks.python import vision


@dataclass
class Config:
    video_path: str = "Data/Front_side_view.mp4"
    model_path: str = "pose_landmarker_heavy.task"

    # outputs
    frame_csv: str = "pose_frames.csv"
    event_csv: str = "pose_events.csv"

    # optional (if you later want overlay)
    out_video: str = ""  # e.g., "elite_analysis_report.mp4"

    # thresholds (same logic as your code)
    elite_threshold: float = 0.18
    overstride_threshold: float = 0.4
    impact_gate: float = 0.4  # prev_z < 0.4 조건

    # processing (optional downsample)
    target_fps: float = 30.0


def make_pose_landmarker(model_path: str):
    # CPU delegate 명시 (가능하면 안정적)
    base_options = python.BaseOptions(
        model_asset_path=model_path,
        delegate=python.BaseOptions.Delegate.CPU
    )
    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    return vision.PoseLandmarker.create_from_options(options)


def main():
    cfg = Config()

    cap = cv2.VideoCapture(cfg.video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {cfg.video_path}")

    src_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    stride = max(1, int(round(src_fps / cfg.target_fps))) if cfg.target_fps > 0 else 1
    proc_fps = src_fps / stride

    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    landmarker = make_pose_landmarker(cfg.model_path)

    frame_rows = []
    event_rows = []

    frame_src = -1
    used_idx = 0

    # impact detection state (same as your logic)
    prev_z = 999.0
    is_descending = False
    last_impact_z = np.nan

    print("🏃‍♂️ 프레임 CSV + 이벤트 CSV 저장 시작 (Tasks + mp.Image)...")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_src += 1
        if frame_src % stride != 0:
            continue

        t_sec = used_idx / proc_fps
        t_ms = int(t_sec * 1000.0)
        used_idx += 1

        # ✅ 네가 원하는 스타일: mediapipe as mp 사용
        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        )

        res = landmarker.detect_for_video(mp_image, t_ms)

        row = {
            "frame_i": used_idx - 1,     # processed frame index
            "frame_src": frame_src,      # original source frame index
            "t_sec": float(t_sec),
            "detected": 0,
        }

        # 33 landmarks wide columns
        for k in range(33):
            row[f"lm{k:02d}_x_px"] = np.nan
            row[f"lm{k:02d}_y_px"] = np.nan
            row[f"lm{k:02d}_z"] = np.nan
            row[f"lm{k:02d}_vis"] = np.nan

        # derived
        row["z_norm"] = np.nan
        row["impact_flag"] = 0
        row["impact_grade"] = None

        is_impact_frame = False
        impact_grade = None
        curr_z = np.nan

        if res.pose_landmarks:
            lm = res.pose_landmarks[0]
            row["detected"] = 1

            for k in range(min(33, len(lm))):
                row[f"lm{k:02d}_x_px"] = float(lm[k].x * w)
                row[f"lm{k:02d}_y_px"] = float(lm[k].y * h)
                row[f"lm{k:02d}_z"] = float(getattr(lm[k], "z", np.nan))
                # tasks landmark may or may not have visibility
                row[f"lm{k:02d}_vis"] = float(getattr(lm[k], "visibility", np.nan))

            # -----------------------------
            # z_norm (your same logic)
            # shoulders: 11,12 / hips: 23,24 / heel: 29 (LEFT_HEEL)
            # -----------------------------
            shoulder_z = (float(lm[11].z) + float(lm[12].z)) / 2.0
            hip_z = (float(lm[23].z) + float(lm[24].z)) / 2.0
            torso_z = abs(shoulder_z - hip_z)

            if torso_z > 1e-8:
                curr_z = abs(float(lm[23].z) - float(lm[29].z)) / torso_z
            else:
                curr_z = 0.0

            row["z_norm"] = float(curr_z)

            # -----------------------------
            # impact detection (your same logic)
            # -----------------------------
            if curr_z < prev_z:
                is_descending = True
            elif curr_z > prev_z and is_descending:
                if prev_z < cfg.impact_gate:
                    last_impact_z = float(prev_z)
                    is_impact_frame = True

                    if last_impact_z <= cfg.elite_threshold:
                        impact_grade = "ELITE"
                    elif last_impact_z >= cfg.overstride_threshold:
                        impact_grade = "OVERSTRIDE"
                    else:
                        impact_grade = "NORMAL"

                is_descending = False

            prev_z = float(curr_z)

        # mark frame row
        if is_impact_frame:
            row["impact_flag"] = 1
            row["impact_grade"] = impact_grade

            # event row
            event_rows.append({
                "event_id": len(event_rows),
                "event_type": "IMPACT",
                "frame_i": int(row["frame_i"]),
                "frame_src": int(row["frame_src"]),
                "t_sec": float(row["t_sec"]),
                "impact_z": float(last_impact_z),
                "grade": impact_grade,
                "z_norm_at_frame": float(row["z_norm"]) if np.isfinite(row["z_norm"]) else np.nan,
            })

        frame_rows.append(row)

        if used_idx % 200 == 0:
            print(f"Processing... processed_frames={used_idx}")

    cap.release()
    landmarker.close()

    # save CSV
    df_frames = pd.DataFrame(frame_rows)
    df_frames.to_csv(cfg.frame_csv, index=False, encoding="utf-8-sig")
    print(f"✅ saved frame csv: {cfg.frame_csv} (rows={len(df_frames)})")

    df_events = pd.DataFrame(event_rows)
    df_events.to_csv(cfg.event_csv, index=False, encoding="utf-8-sig")
    print(f"✅ saved event csv: {cfg.event_csv} (events={len(df_events)})")


if __name__ == "__main__":
    main()


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1770622398.440112 2102181 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1770622398.536813 2102193 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


🏃‍♂️ 프레임 CSV + 이벤트 CSV 저장 시작 (Tasks + mp.Image)...


W0000 00:00:1770622399.394678 2102180 landmark_projection_calculator.cc:78] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


✅ saved frame csv: pose_frames.csv (rows=165)
✅ saved event csv: pose_events.csv (events=13)


## 제미나이 API 모델 리스트

In [ ]:
import google.generativeai as genai

# 본인의 API 키를 설정하세요
genai.configure(api_key="GEMENI_API_KEY")

# 사용 가능한 모델 리스트 출력
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(f"모델 이름: {m.name}")
        print(f"설명: {m.description}")
        print("-" * 30)

/mnt/home_dnlab/shkim/Running_FeedBack_Model/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/tmp/ipykernel_2099797/3815708470.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


모델 이름: models/gemini-2.5-flash
설명: Stable version of Gemini 2.5 Flash, our mid-size multimodal model that supports up to 1 million tokens, released in June of 2025.
------------------------------
모델 이름: models/gemini-2.5-pro
설명: Stable release (June 17th, 2025) of Gemini 2.5 Pro
------------------------------
모델 이름: models/gemini-2.0-flash
설명: Gemini 2.0 Flash
------------------------------
모델 이름: models/gemini-2.0-flash-001
설명: Stable version of Gemini 2.0 Flash, our fast and versatile multimodal model for scaling across diverse tasks, released in January of 2025.
------------------------------
모델 이름: models/gemini-2.0-flash-exp-image-generation
설명: Gemini 2.0 Flash (Image Generation) Experimental
------------------------------
모델 이름: models/gemini-2.0-flash-lite-001
설명: Stable version of Gemini 2.0 Flash-Lite
------------------------------
모델 이름: models/gemini-2.0-flash-lite
설명: Gemini 2.0 Flash-Lite
------------------------------
모델 이름: models/gemini-exp-1206
설명: Experimental releas

## 제미나이 API 사용하여 csv 분석

## 팔꿈치 각도 실제 계산 -> 실제 분석 알고리즘 (v0)

In [ ]:
import pandas as pd
import numpy as np
import google.generativeai as genai

# Gemini 설정
genai.configure(api_key="GEMENI_API_KEY")
model = genai.GenerativeModel('gemini-2.5-flash')

def get_combined_stats(frame_path, event_path):
    # 1. 데이터 로드
    df_frame = pd.read_csv(frame_path)
    df_event = pd.read_csv(event_path)
    
    # 2. 프레임 기반 분석: 팔꿈치 각도 및 수직 진폭
    # 팔꿈치 각도 계산 (임의의 간단한 벡터 각도 공식 적용)
    def calc_angle(a, b, c):
        ba = a - b
        bc = c - b
        denom = (np.linalg.norm(ba) * np.linalg.norm(bc)) + 1e-8
        cosine_angle = np.dot(ba, bc) / denom
        return np.degrees(np.arccos(np.clip(cosine_angle, -1.0, 1.0)))

    # 평균 팔꿈치 각도 (RIGHT only: 12-14-16)
    # pose_frames.csv에 있는 lm12/lm14/lm16 좌표(px)를 이용해 프레임별 각도 계산 후 평균
    x12, y12 = df_frame['lm12_x_px'].to_numpy(), df_frame['lm12_y_px'].to_numpy()  # R shoulder
    x14, y14 = df_frame['lm14_x_px'].to_numpy(), df_frame['lm14_y_px'].to_numpy()  # R elbow
    x16, y16 = df_frame['lm16_x_px'].to_numpy(), df_frame['lm16_y_px'].to_numpy()  # R wrist

    elbow_angles = []
    for i in range(len(df_frame)):
        if np.any(np.isnan([x12[i], y12[i], x14[i], y14[i], x16[i], y16[i]])):
            continue
        a = np.array([x12[i], y12[i]], dtype=float)
        b = np.array([x14[i], y14[i]], dtype=float)
        c = np.array([x16[i], y16[i]], dtype=float)

        ang = calc_angle(a, b, c)
        if np.isfinite(ang):
            elbow_angles.append(ang)

    elbow_angle_avg = float(np.mean(elbow_angles)) if len(elbow_angles) > 0 else np.nan
    
    # 수직 진폭 (코 lm00_y_px 기준)
    v_oscillation = df_frame['lm00_y_px'].max() - df_frame['lm00_y_px'].min()
    
    # 3. 이벤트 기반 분석: 오버스트라이드 및 비대칭
    # 이벤트 파일은 이미 착지 시점의 impact_z가 계산되어 있음
    avg_impact_z = df_event['impact_z'].mean()
    # 홀수/짝수 인덱스를 활용한 좌우 비대칭 계산 (데이터 순서 기준)
    left_events = df_event.iloc[0::2]['impact_z']
    right_events = df_event.iloc[1::2]['impact_z']
    asymmetry = abs(left_events.mean() - right_events.mean()) / avg_impact_z * 100
    
    # 4. 케이던스 계산
    duration = df_frame['t_sec'].max() - df_frame['t_sec'].min()
    step_count = len(df_event)
    cadence = (step_count / duration) * 60
    
    return {
        "cadence": round(cadence, 1),
        "v_oscillation": round(v_oscillation, 1),
        "avg_impact_z": round(avg_impact_z, 3),
        "asymmetry": round(asymmetry, 1),
        "elbow_angle": round(elbow_angle_avg, 1)
    }

def generate_integrated_report(stats):
    prompt = f"""
    당신은 데이터 기반의 생체역학 마라톤 코치입니다. 
    '전체 프레임 기록'과 '착지 이벤트 데이터'를 통합 분석한 결과입니다. 
    이 러너의 마라톤 완주 가능성과 자세 결함을 분석하십시오.

    [통합 데이터 지표]
    - 평균 케이던스: {stats['cadence']} SPM
    - 수직 진폭(상하 출렁임): {stats['v_oscillation']} px
    - 평균 오버스트라이드(Impact Z): {stats['avg_impact_z']} (0.2 미만 권장)
    - 좌우 착지 비대칭률: {stats['asymmetry']}%
    - 평균 팔꿈치 각도: {stats['elbow_angle']}도

    [분석 필수 포인트]
    1. **팔꿈치-케이던스 협응:** 고정된 팔꿈치 각도가 현재의 {stats['cadence']} SPM 리듬을 얼마나 잘 지지하고 있는가?
    2. **오버스트라이드 진단:** 이벤트 데이터에서 나타난 Impact Z 수치가 지면 제동(Braking force)을 얼마나 억제하고 있는가?
    3. **에너지 효율:** 수직 진폭 수치가 마라톤 후반부 근피로에 미치는 영향.
    4. **비대칭 경고:** {stats['asymmetry']}%의 비대칭이 30km 지점 이후 부상(장경인대, 발목 등)으로 이어질 시나리오.
    5. **마라톤 팁:** 이 러너가 풀코스에서 '서브 3' 혹은 '완주'를 위해 반드시 고쳐야 할 가장 치명적인 포인트 1가지.

    [출력 스타일]
    - 데이터 중심의 냉철한 분석 후, 따뜻한 코칭 제언으로 마무리할 것.
    - 사용자가 이해하기 쉬운 언어로 설명할 것 (예: "당신의 팔꿈치가 너무 펴져 있어요" 대신 "팔꿈치가 너무 펴져 있어서...").
    - 사용자에게 기술적 분석과 함께 실천 가능한 조언을 제공할 것.
    - 불필요한 서론 없이 즉시 리포트 시작.
    """
    
    response = model.generate_content(prompt)
    return response.text

# 실행
stats = get_combined_stats('pose_frames.csv', 'pose_events.csv')
print(generate_integrated_report(stats))


## 러너 마라톤 완주 가능성 및 자세 결함 분석 리포트

종합적인 데이터 분석 결과, 현재 러너님의 자세는 마라톤 완주에 **잠재적 위험 요소를 안고 있으며, 특히 서브 3 달성에는 상당한 개선이 필요합니다.** 일부 긍정적인 지표도 있지만, 마라톤 후반부의 에너지 효율과 부상 위험 관리가 핵심 과제가 될 것입니다.

---

### **[기술적 분석]**

1.  **팔꿈치-케이던스 협응: 142.7 SPM & 평균 팔꿈치 각도 89.0도**
    *   **분석:** 팔꿈치 각도 89.0도는 일반적으로 이상적인 90도에 매우 근접하여 팔 스윙의 기본적인 자세는 안정적입니다. 하지만 현재의 평균 케이던스 142.7 SPM은 마라톤 러닝에 있어 매우 낮은 수치에 해당합니다. 팔꿈치 각도 자체는 효율적인 팔 스윙을 위한 좋은 기반을 제공할 수 있으나, 이 각도가 **낮은 케이던스(142.7 SPM)**를 끌어올리는 데 효과적으로 기여하지 못하고 있습니다. 즉, 팔 스윙의 '크기'는 좋지만 '속도'가 부족하여 전체적인 러닝 리듬을 충분히 지지하지 못하는 상황으로 보입니다. 이는 다리 동작의 보조력 약화로 이어질 수 있습니다.

2.  **오버스트라이드 진단: Impact Z 0.14**
    *   **분석:** 평균 오버스트라이드(Impact Z) 수치 0.14는 권장치(0.2 미만)보다 낮은 매우 긍정적인 결과입니다. 이는 러너님이 착지 시 발이 몸의 중심선보다 과도하게 앞서 나가지 않도록 효과적으로 제어하고 있음을 의미합니다. 지면 제동(Braking force)이 상당히 억제되어 있어, 착지 충격으로 인한 속도 감소 및 무릎/정강이 부하를 줄이는 데 유리합니다. 이 부분은 러너님의 강력한 장점 중 하나입니다.

3.  **에너지 효율: 수직 진폭 141.2 px**
    *   **분석:** 수직 진폭 141.2 px는 상대적으로 높은 수치로 판단됩니다. 이는 러닝 시 몸이 상하로 불필요하게 많이 움직인다는 것을 의미하며, 앞으로 나아가는 데 사용되

## csv 수치 분석

In [6]:
import pandas as pd
import numpy as np

def analyze_biomechanics(csv_path):
    df = pd.read_csv(csv_path)
    
    # 1. 팔꿈치 각도 계산 (어깨-팔꿈치-손목 벡터 내적)
    def calculate_angle(a, b, c):
        ba = a - b
        bc = c - b
        cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
        return np.degrees(np.arccos(np.clip(cosine_angle, -1.0, 1.0)))

    # 예시: 왼쪽 팔꿈치 각도 산출 (lm11: 어깨, lm13: 팔꿈치, lm15: 손목)
    df['elbow_angle'] = df.apply(lambda row: calculate_angle(
        np.array([row['lm11_x_px'], row['lm11_y_px']]),
        np.array([row['lm13_x_px'], row['lm13_y_px']]),
        np.array([row['lm15_x_px'], row['lm15_y_px']])
    ), axis=1)

    # 2. 수직 진폭 (Vertical Oscillation)
    # 골반(lm23, lm24)의 중심 y좌표 변화량 측정
    df['hip_center_y'] = (df['lm23_y_px'] + df['lm24_y_px']) / 2
    v_oscillation = df['hip_center_y'].max() - df['hip_center_y'].min()

    # 3. 케이던스 (Cadence)
    # 발목(lm27, lm28) y좌표의 로컬 미니멈(착지) 주기를 분석
    y_signal = df['lm27_y_px'].values
    peaks = (np.diff(np.sign(np.diff(y_signal))) > 0).nonzero()[0] + 1
    total_time = df['t_sec'].max() - df['t_sec'].min()
    cadence = (len(peaks) / total_time) * 60

    # 4. 오버스트라이드 (Impact Z)
    # 착지 시점(발목 y좌표가 최대일 때) 발목과 골반의 z축 거리 차이
    impact_frames = df.iloc[peaks]
    avg_impact_z = (impact_frames['lm27_z'] - impact_frames['lm23_z']).abs().mean()

    # 5. 좌우 비대칭률 (Asymmetry)
    # 왼쪽/오른쪽 발목의 최고점 도달 시간차 또는 y좌표 이동 범위 차이 계산
    left_range = df['lm27_y_px'].max() - df['lm27_y_px'].min()
    right_range = df['lm28_y_px'].max() - df['lm28_y_px'].min()
    asymmetry = abs(left_range - right_range) / ((left_range + right_range) / 2) * 100

    return {
        "평균 케이던스": round(cadence, 1),
        "수직 진폭 (px)": round(v_oscillation, 1),
        "평균 오버스트라이드 (Impact Z)": round(avg_impact_z, 3),
        "좌우 비대칭률 (%)": round(asymmetry, 1),
        "평균 팔꿈치 각도 (도)": round(df['elbow_angle'].mean(), 1)
    }

# 실행
stats = analyze_biomechanics('pose_frames.csv')
for key, value in stats.items():
    print(f"{key}: {value}")

평균 케이던스: 230.5
수직 진폭 (px): 182.1
평균 오버스트라이드 (Impact Z): 0.162
좌우 비대칭률 (%): 0.1
평균 팔꿈치 각도 (도): 122.0


## pose_frames.csv만 제미나이 분석

In [ ]:
import pandas as pd
import numpy as np
import google.generativeai as genai

# Gemini 설정 (보안을 위해 API 키는 환경변수 권장)
genai.configure(api_key="GEMENI_API_KEY")
model = genai.GenerativeModel('gemini-2.5-flash') # 모델명 확인 필요 (2.5는 현재 존재하지 않음)

def get_stats_from_frames(frame_path):
    # 1. 데이터 로드
    df_frame = pd.read_csv(frame_path)
    
    # 2. 수직 진폭 (코 lm00_y_px 기준)
    v_oscillation = df_frame['lm00_y_px'].max() - df_frame['lm00_y_px'].min()
    
    # 3. 팔꿈치 각도 (가상의 계산 로직 또는 평균값)
    # 실제로는 x,y 좌표로 계산해야 함. 여기서는 테스트용 고정값
    elbow_angle_avg = 92.5 
    
    # 4. 프레임 데이터를 통한 케이던스 추정 (임시)
    # y축(코/골반)의 피크 개수를 세어 걸음 수를 추정할 수 있으나, 
    # 테스트를 위해 임의의 로직 적용
    duration = df_frame['t_sec'].max() - df_frame['t_sec'].min()
    estimated_steps = len(df_frame) / 30 # 초당 30프레임 가정 시 샘플링
    cadence = (estimated_steps / duration) * 60 if duration > 0 else 170

    # 5. 이벤트 데이터 부재에 따른 더미(Dummy) 데이터
    # 프레임만 있을 때는 계산이 어려우므로 분석을 위해 임시값 부여
    avg_impact_z = 0.25  # 약간 높은 수치 가정
    asymmetry = 3.2      # 약간의 비대칭 가정

    return {
        "cadence": round(cadence, 1),
        "v_oscillation": round(v_oscillation, 1),
        "avg_impact_z": round(avg_impact_z, 3),
        "asymmetry": round(asymmetry, 1),
        "elbow_angle": round(elbow_angle_avg, 1)
    }

def generate_integrated_report(stats):
    prompt = f"""
    당신은 데이터 기반의 생체역학 마라톤 코치입니다. 
    분석된 지표를 바탕으로 러너의 마라톤 완주 가능성과 자세 결함을 분석하십시오.

    [데이터 지표]
    - 평균 케이던스: {stats['cadence']} SPM
    - 수직 진폭: {stats['v_oscillation']} px
    - 평균 오버스트라이드(Impact Z): {stats['avg_impact_z']} (0.2 미만 권장)
    - 좌우 착지 비대칭률: {stats['asymmetry']}%
    - 평균 팔꿈치 각도: {stats['elbow_angle']}도

    [분석 필수 포인트]
    1. 팔꿈치-케이던스 협응 분석
    2. 오버스트라이드(Impact Z) 진단
    3. 수직 진폭과 에너지 효율 관계
    4. 비대칭 경고 및 부상 시나리오
    5. 풀코스 완주를 위한 가장 치명적인 포인트 1가지 교정 제언

    [출력 스타일]
    - 냉철한 데이터 분석 후 따뜻한 코칭으로 마무리.
    """
    
    response = model.generate_content(prompt)
    return response.text

# 실행 (frames 파일만 사용)
try:
    stats = get_stats_from_frames('pose_frames.csv')
    print(generate_integrated_report(stats))
except Exception as e:
    print(f"오류 발생: {e}")

분석된 지표를 바탕으로 러너의 마라톤 완주 가능성과 자세 결함을 분석합니다.

---

### **[데이터 기반 분석 보고서]**

러너님의 현재 데이터 지표는 마라톤 완주를 위한 여러 가지 개선점이 명확하게 드러나고 있습니다. 특히, 현재의 자세는 에너지 효율성 측면에서 매우 불리하며, 장거리 주행 시 부상 위험을 높이는 치명적인 요소들을 포함하고 있습니다. 냉철하게 분석하겠습니다.

**1. 평균 케이던스: 60.4 SPM**
*   **분석:** 이 수치는 러닝 케이던스로서는 비정상적으로 낮은 수준입니다. 일반적인 효율적인 러닝 케이던스는 최소 160 SPM 이상, 이상적으로는 170-180 SPM 이상을 권장합니다. 60.4 SPM은 걷거나 매우 느린 조깅 수준에 해당하며, 러닝으로서는 추진력을 상실하고 비효율적인 움직임이 발생할 수밖에 없는 수치입니다. 이 데이터가 오독되었거나 측정이 잘못된 것이 아니라면, 러너님은 사실상 '달린다'기보다는 '빠르게 걷는' 수준에 가깝다고 볼 수 있습니다.

**2. 수직 진폭: 141.2 px**
*   **분석:** 수직 진폭의 단위가 'px'로 제시되어 절대적인 물리량으로 해석하기는 어렵습니다만, 일반적인 센서 데이터에서 높은 'px' 값은 러닝 시 불필요한 상하 움직임이 크다는 것을 의미합니다. 이는 에너지를 앞으로 나아가는 데 사용하기보다 몸을 위로 띄우는 데 낭비하고 있다는 명확한 신호입니다. 높은 수직 진폭은 에너지 효율을 극도로 떨어뜨려 조기 피로를 유발하고, 지면에 가해지는 충격을 증가시켜 부상 위험을 높입니다.

**3. 평균 오버스트라이드(Impact Z): 0.25 (0.2 미만 권장)**
*   **분석:** 권장 기준(0.2 미만)을 0.05 초과하는 0.25는 명백한 오버스트라이드(Overstride)를 나타냅니다. 발이 몸의 무게 중심보다 훨씬 앞에 착지한다는 의미입니다. 이는 러닝 효율을 떨어뜨리는 가장 치명적인 요소 중 하나입니다. 지면에 착지할 때 제동력이 발생하여 앞으로 나아가야 할